## Weight Optimizer
Learns per-indicator weights that maximize AUC against realized SPY drawdown labels.
Uses SLSQP with 5-fold time series CV and L2 regularization toward equal weights to minimize the generalization gap.

In [ ]:
import os
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.optimize import minimize
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import TimeSeriesSplit

N_JOBS = os.cpu_count()
print(f"using {N_JOBS} cores")

### 1. Load data

In [ ]:
# assumes working directory is project root (default in VS Code)
try:
    df = pd.read_parquet("data/processed/stress_score.parquet")
    DOCS_PATH = Path("docs")
except FileNotFoundError:
    df = pd.read_parquet("../data/processed/stress_score.parquet")
    DOCS_PATH = Path("../docs")

df.index = pd.to_datetime(df.index)

INDICATOR_COLS = [
    "T10Y2Y",
    "T10Y3M",
    "T30Y10Y",
    "USALOLITOAASTSAM",
    "UMCSENT",
    "PERMIT",
    "NEWORDER",
    "ICSA",
    "DRCCLACBS",
    "BAMLH0A0HYM2",
    "XLK_XLV",
    "TLT",
    "HG=F",
    "CL=F",
    "EEM",
    "DX=F",
]

df = df.dropna(subset=INDICATOR_COLS)
print(f"{len(df)} rows, {df.index[0].date()} to {df.index[-1].date()}")

### 2. Build drawdown labels

In [ ]:
DRAWDOWN_THRESHOLD = 0.10

drawdown = df["SPY"] / df["SPY"].cummax() - 1
labels = (drawdown <= -DRAWDOWN_THRESHOLD).astype(int)

print(f"drawdown days: {labels.sum()} of {len(labels)} ({labels.mean():.1%})")

fig, ax = plt.subplots(figsize=(14, 3))
ax.fill_between(
    drawdown.index, drawdown, 0, where=drawdown < 0, alpha=0.4, color="#d62728"
)
ax.axhline(
    -DRAWDOWN_THRESHOLD,
    color="black",
    linewidth=0.8,
    linestyle="--",
    label=f"{DRAWDOWN_THRESHOLD:.0%} threshold",
)
ax.set_ylabel("drawdown")
ax.set_title("SPY drawdown from expanding high")
ax.legend()
plt.tight_layout()
plt.show()

### 3. Optimization functions
SLSQP accepts an `alpha` parameter for L2 regularization toward equal weights.
The penalty term is `alpha * sum((w - 1/N)^2)`, which shrinks extreme weights
back toward the equal-weight baseline and reduces overfitting to the training period.

In [ ]:
N = len(INDICATOR_COLS)
EQUAL_WEIGHTS = np.ones(N) / N

# bounds: small negatives allowed so an indicator can slightly reduce the score,
# cap at 0.25 to prevent any single indicator from dominating.
# feasibility of sum=1: min sum = 16*(-0.10) = -1.60, max = 16*0.25 = 4.00, so sum=1 is reachable.
BOUNDS = [(-0.10, 0.25)] * N

# sum-to-one is still enforced: the regularization term alpha*(w - 1/N)^2 anchors
# weights near 1/N, which only makes sense if they sum to 1.
SUM_TO_ONE = [{"type": "eq", "fun": lambda w: w.sum() - 1}]


def soft_auc(weights, X, y):
    """Differentiable AUC approximation using sigmoid on pairwise differences.

    Steepness=10: with pre-normalized 0-1 features and weights summing to 1,
    pairwise score differences fall in [-1, 1]. At steepness=10 a difference
    of 0.1 maps to sigmoid(1.0) ~ 0.73, giving sharp enough discrimination
    without saturating the gradient on small differences.
    """
    score = (X * weights).sum(axis=1)
    pos = score[y == 1].values
    neg = score[y == 0].values
    diff = pos[:, None] - neg[None, :]
    return -np.mean(1 / (1 + np.exp(-diff * 10)))


def fit_slsqp(X_train, y_train, alpha=0.0):
    def objective(weights):
        reg = alpha * ((weights - EQUAL_WEIGHTS) ** 2).sum()
        return soft_auc(weights, X_train, y_train) + reg

    result = minimize(
        objective,
        EQUAL_WEIGHTS,
        method="SLSQP",
        bounds=BOUNDS,
        constraints=SUM_TO_ONE,
        options={"ftol": 1e-9, "maxiter": 1000},
    )
    return result.x

### 4. CV helper
Runs 5-fold time series CV for a given method and alpha.
Returns per-fold train AUC, test AUC, and the generalization gap (train - test).

In [ ]:
def run_cv(fit_fn, alpha=0.0, n_splits=5, X=None, y=None):
    # X and y are accepted explicitly so this function is safe to call
    # from joblib workers without relying on notebook globals.
    if X is None:
        X = df[INDICATOR_COLS]
    if y is None:
        y = labels

    tscv = TimeSeriesSplit(n_splits=n_splits)
    folds = []

    for fold, (train_idx, test_idx) in enumerate(tscv.split(X)):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        if y_test.nunique() < 2 or y_train.nunique() < 2:
            continue

        weights = fit_fn(X_train, y_train, alpha=alpha)
        train_auc = roc_auc_score(y_train, (X_train * weights).sum(axis=1))
        test_auc = roc_auc_score(y_test, (X_test * weights).sum(axis=1))

        folds.append(
            {
                "fold": fold + 1,
                "train_size": len(train_idx),
                "test_size": len(test_idx),
                "test_start": X.index[test_idx[0]].date(),
                "test_end": X.index[test_idx[-1]].date(),
                "train_auc": train_auc,
                "test_auc": test_auc,
                "gap": train_auc - test_auc,
                "weights": weights,
            }
        )

    return folds

### 5. Baseline CV (alpha=0)

In [ ]:
def fit_equal(X_train, y_train, alpha=0.0):
    # alpha unused; equal weights don't depend on regularization
    return EQUAL_WEIGHTS


def summarize(folds, label):
    test_aucs = [f["test_auc"] for f in folds]
    gaps = [f["gap"] for f in folds]
    print(
        f"{label:14s}  test AUC: {np.mean(test_aucs):.4f} +/- {np.std(test_aucs):.4f}  gap: {np.mean(gaps):.4f} +/- {np.std(gaps):.4f}"
    )


X_data = df[INDICATOR_COLS]
y_data = labels

folds_equal = run_cv(fit_equal, X=X_data, y=y_data)

print("fold sizes:")
for f in folds_equal:
    print(
        f"  fold {f['fold']}: train={f['train_size']}  test={f['test_size']}"
        f"  ({f['test_start']} to {f['test_end']})"
    )
print()

folds_slsqp = run_cv(fit_slsqp, alpha=0.0, X=X_data, y=y_data)

summarize(folds_equal, "equal")
summarize(folds_slsqp, "slsqp")

In [ ]:
fold_nums = [f["fold"] for f in folds_slsqp]
train_aucs = [f["train_auc"] for f in folds_slsqp]
test_aucs = [f["test_auc"] for f in folds_slsqp]
equal_test_aucs = [f["test_auc"] for f in folds_equal]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fold_nums, train_aucs, "o--", color="#1f77b4", label="train AUC")
ax.plot(fold_nums, test_aucs, "o-", color="#1f77b4", alpha=0.5, label="test AUC")
ax.fill_between(
    fold_nums, test_aucs, train_aucs, alpha=0.1, color="#1f77b4", label="gap"
)
ax.plot(
    fold_nums,
    equal_test_aucs,
    "o:",
    color="#888888",
    alpha=0.7,
    label="equal weight test AUC",
)
ax.set_title("SLSQP (alpha=0): train vs test AUC")
ax.set_xlabel("fold")
ax.set_ylabel("AUC")
ax.set_ylim(0.5, 1.05)
ax.legend()
plt.tight_layout()
plt.show()

### 6. Alpha sweep
Sweeps regularization strength for SLSQP.
Goal: find the alpha that maximizes mean test AUC while keeping the gap small.

In [ ]:
# Log-ish grid covering 0-10; spacing is denser at low alpha where behavior changes fastest.
ALPHAS = [0.0, 0.1, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.5, 10.0]


def _sweep_job(alpha, X, y):
    folds = run_cv(fit_slsqp, alpha=alpha, X=X, y=y)
    return {
        "alpha": alpha,
        "mean_test_auc": np.mean([f["test_auc"] for f in folds]),
        "mean_gap": np.mean([f["gap"] for f in folds]),
        "std_test_auc": np.std([f["test_auc"] for f in folds]),
    }


sweep_raw = Parallel(n_jobs=N_JOBS)(
    delayed(_sweep_job)(alpha, X_data, y_data) for alpha in ALPHAS
)

sweep_results = sorted(sweep_raw, key=lambda r: r["alpha"])
print("sweep complete")

In [ ]:
alphas = [r["alpha"] for r in sweep_results]
test_aucs = [r["mean_test_auc"] for r in sweep_results]
gaps = [r["mean_gap"] for r in sweep_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(alphas, test_aucs, "o-", color="#1f77b4", label="slsqp")
axes[0].axhline(
    np.mean([f["test_auc"] for f in folds_equal]),
    color="grey",
    linewidth=0.8,
    linestyle="--",
    label="equal weight",
)
axes[0].set_xlabel("alpha")
axes[0].set_ylabel("mean test AUC")
axes[0].set_title("test AUC vs regularization strength")
axes[0].legend()

axes[1].plot(alphas, gaps, "o-", color="#1f77b4", label="slsqp")
axes[1].axhline(0, color="grey", linewidth=0.8, linestyle="--")
axes[1].set_xlabel("alpha")
axes[1].set_ylabel("mean gap (train - test)")
axes[1].set_title("generalization gap vs regularization strength")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Penalize the gap rather than maximizing test AUC alone. Pure max-test-AUC tends to pick
# alphas where one fold happened to score well but the gap is still large; subtracting
# abs(gap) rewards alphas where performance is both high and consistent across folds.
# The commented alternative is useful if you want to ignore variance and just see peak AUC.
# best = max(sweep_results, key=lambda r: r["mean_test_auc"])
best = max(sweep_results, key=lambda r: r["mean_test_auc"] - abs(r["mean_gap"]))
print(
    f"slsqp: best alpha={best['alpha']:.2f}  test AUC={best['mean_test_auc']:.4f}  gap={best['mean_gap']:.4f}"
)

### 7. Final CV with best alpha

In [ ]:
# best = max(sweep_results, key=lambda r: r["mean_test_auc"])
BEST_ALPHA = best["alpha"]
print(f"best alpha: {BEST_ALPHA}  (test AUC {best['mean_test_auc']:.4f})")

folds_slsqp_reg = run_cv(fit_slsqp, alpha=BEST_ALPHA, X=X_data, y=y_data)

print("final CV results with tuned alpha:")
summarize(folds_equal, "equal")
summarize(folds_slsqp_reg, f"slsqp a={BEST_ALPHA}")

### 8. Weight stability (best alpha)

In [ ]:
weights_per_fold = np.vstack([f["weights"] for f in folds_slsqp_reg])
mean_w = weights_per_fold.mean(axis=0)
std_w = weights_per_fold.std(axis=0)
order = np.argsort(mean_w)[::-1]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    np.array(INDICATOR_COLS)[order],
    mean_w[order],
    xerr=std_w[order],
    color="#1f77b4",
    alpha=0.7,
    capsize=3,
)
ax.axvline(1 / N, color="grey", linewidth=0.8, linestyle="--", label="equal weight")
ax.set_title(f"SLSQP (alpha={BEST_ALPHA}): mean weight across folds (+/- 1 std)")
ax.set_xlabel("weight")
ax.legend()
plt.tight_layout()
fig.savefig(DOCS_PATH / "weight_stability.png", dpi=150, bbox_inches="tight")
plt.show()

### 9. Final model (full dataset)

In [ ]:
# train on full data for final weights -- in-sample AUC only, not comparable to CV results
X_full = df[INDICATOR_COLS]
y_full = labels

final_weights = fit_slsqp(X_full, y_full, alpha=BEST_ALPHA)

score_equal = (X_full * EQUAL_WEIGHTS).sum(axis=1)
score_slsqp = (X_full * final_weights).sum(axis=1)

print("in-sample AUC (full data, for reference only):")
print(f"  equal:  {roc_auc_score(y_full, score_equal):.4f}")
print(f"  slsqp:  {roc_auc_score(y_full, score_slsqp):.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

for score, label, color in [
    (
        score_equal,
        f"equal weight (AUC {roc_auc_score(y_full, score_equal):.3f})",
        "#aec7e8",
    ),
    (score_slsqp, f"SLSQP (AUC {roc_auc_score(y_full, score_slsqp):.3f})", "#1f77b4"),
]:
    fpr, tpr, _ = roc_curve(y_full, score)
    ax.plot(fpr, tpr, label=label)

ax.plot([0, 1], [0, 1], color="grey", linewidth=0.8, linestyle="--")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("ROC curves (full data, in-sample)")
ax.legend()
plt.tight_layout()
fig.savefig(DOCS_PATH / "roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()

### 10. Optimized score vs SPY and drawdown

In [ ]:
STRESS_PERIODS = [
    ("2007-10-01", "2009-03-01", "GFC"),
    ("2011-07-01", "2011-10-01", "EU debt crisis"),
    ("2015-08-01", "2016-02-01", "China slowdown"),
    ("2018-10-01", "2018-12-31", "Q4 selloff"),
    ("2020-02-01", "2020-04-01", "COVID crash"),
    ("2022-01-01", "2022-10-01", "Rate hike cycle"),
    ("2025-01-20", "2025-04-01", "Tariff shock"),
]

fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.fill_between(
    df.index, score_slsqp, alpha=0.35, color="#d62728", label="Optimized score"
)
ax1.plot(df.index, score_slsqp, color="#d62728", linewidth=0.6)
ax1.plot(
    df.index,
    score_equal,
    color="#888888",
    linewidth=0.8,
    linestyle="--",
    label="Equal weight",
)
ax2.plot(df.index, df["SPY"], color="#1f77b4", linewidth=1.0, label="SPY")

for start, end, label in STRESS_PERIODS:
    ax1.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08, color="grey")
    mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
    ax1.text(
        mid,
        0.97,
        label,
        ha="center",
        va="top",
        fontsize=7.5,
        color="#444444",
        transform=ax1.get_xaxis_transform(),
    )

ax1.set_ylabel("Stress score", color="#d62728")
ax2.set_ylabel("SPY", color="#1f77b4")
ax1.set_ylim(0, 1)
ax1.set_yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax1.yaxis.grid(True, linewidth=0.3, color="grey", alpha=0.25)
ax1.set_axisbelow(True)
ax2.yaxis.grid(False)
ax1.tick_params(axis="y", colors="#d62728")
ax2.tick_params(axis="y", colors="#1f77b4")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax1.xaxis.set_major_locator(mdates.YearLocator())

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.1),
    ncol=3,
)

fig.suptitle("Optimized Stress Score vs SPY", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(
    DOCS_PATH / "optimized_vs_equalweight_spy.png", dpi=150, bbox_inches="tight"
)
plt.show()

In [ ]:
spy_dd = df["SPY"] / df["SPY"].cummax() - 1
spy_dd_norm = (spy_dd - spy_dd.min()) / (spy_dd.max() - spy_dd.min())
spy_dd_norm = 1 - spy_dd_norm  # invert so crashes spike up

fig, ax = plt.subplots(figsize=(14, 5))

ax.fill_between(
    df.index, score_slsqp, alpha=0.3, color="#d62728", label="Optimized score"
)
ax.plot(df.index, score_slsqp, color="#d62728", linewidth=0.6)
ax.plot(
    df.index,
    score_equal,
    color="#888888",
    linewidth=0.8,
    linestyle="--",
    label="Equal weight",
)
ax.fill_between(
    df.index, spy_dd_norm, alpha=0.3, color="#1f77b4", label="SPY drawdown (normalized)"
)
ax.plot(df.index, spy_dd_norm, color="#1f77b4", linewidth=0.6)

for start, end, label in STRESS_PERIODS:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.08, color="grey")
    mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
    ax.text(
        mid,
        0.97,
        label,
        ha="center",
        va="top",
        fontsize=7.5,
        color="#444444",
        transform=ax.get_xaxis_transform(),
    )

ax.set_ylim(0, 1)
ax.set_ylabel("Score (0-1)")
ax.set_yticks([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax.yaxis.grid(True, linewidth=0.3, color="grey", alpha=0.25)
ax.set_axisbelow(True)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=3)
ax.set_title("Optimized Stress Score vs SPY Drawdown")

plt.tight_layout()
fig.savefig(
    DOCS_PATH / "optimized_vs_equalweight_drawdown.png", dpi=150, bbox_inches="tight"
)
plt.show()